# ◈ Insight Lineage — Test Notebook
Tests `insight_lineage_tab.py` logic end-to-end inside Jupyter.  
All graph-building and HTML-rendering code is **copied verbatim** from the
generated tab file so no changes are needed to any deployed file.

---
### How to use
1. Fill in your Greenplum connection details in **Cell 2**.
2. Run all cells (`Kernel → Restart & Run All`).
3. Use the **dropdown widget** to select an insight type.
4. Inspect the ODM rule columns shown in the summary, then click **Show Lineage**.


In [ ]:
# ── Cell 2 : DB Connection Config ──────────────────────────────────────
GP_HOST    = "greenplum-rdsp.zur.swissbank.com"
GP_PORT    = 5432
GP_DB      = "gprdsp"
GP_USER    = "ds_rdsp_dev"
IKG_SCHEMA = "sandbox_prj_smart_insights"

# ── Imports ──────────────────────────────────────────────────────────────
import json, re, os, tempfile, warnings
from pathlib import Path

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, IFrame

try:
    from sqlalchemy import create_engine, text
except ImportError:
    warnings.warn("Install sqlalchemy: pip install sqlalchemy psycopg2-binary")

import getpass
_pw = getpass.getpass(f"Greenplum password for {GP_USER}@{GP_HOST}:")
_engine = create_engine(
    f"postgresql+psycopg2://{GP_USER}:{_pw}@{GP_HOST}:{GP_PORT}/{GP_DB}",
    pool_pre_ping=True,
)

def _run(sql: str) -> pd.DataFrame:
    with _engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

print("✅  DB connection ready")


In [ ]:
# ── Cell 3 : Data-access functions (mirror of utils/data.py) ────────────

def get_all_insight_types() -> pd.DataFrame:
    return _run(f"""
        SELECT DISTINCT insight_type
        FROM {IKG_SCHEMA}.odm_rule_metadata_auto_refresh
        WHERE insight_type IS NOT NULL AND TRIM(insight_type) <> ''
        ORDER BY insight_type
    """)

def get_insight_rule_meta(insight_type: str) -> pd.DataFrame:
    safe = str(insight_type).replace("'", "''")
    return _run(f"""
        SELECT *
        FROM {IKG_SCHEMA}.odm_rule_metadata_auto_refresh
        WHERE insight_type = '{safe}'
    """)

def get_insight_column_lineage(insight_type: str) -> pd.DataFrame:
    safe = str(insight_type).replace("'", "''")
    return _run(f"""
        WITH rule_cols AS (
            SELECT DISTINCT rule_column, target_type
            FROM {IKG_SCHEMA}.odm_rule_metadata_auto_refresh
            WHERE insight_type = '{safe}'
              AND rule_column IS NOT NULL AND TRIM(rule_column) <> ''
        ),
        anchor AS (
            SELECT cl.target_table, cl.target_schema, cl.target_column,
                   cl.source_table, cl.source_schema, cl.source_column,
                   cl.logic, cl.process, cl.path, 0 AS depth
            FROM {IKG_SCHEMA}.ikg_column_lineage_master_auto_refresh cl
            JOIN rule_cols rc ON cl.target_column = rc.rule_column
        ),
        upstream AS (
            SELECT * FROM anchor
            UNION ALL
            SELECT cl.target_table, cl.target_schema, cl.target_column,
                   cl.source_table, cl.source_schema, cl.source_column,
                   cl.logic, cl.process, cl.path, u.depth + 1
            FROM {IKG_SCHEMA}.ikg_column_lineage_master_auto_refresh cl
            JOIN upstream u
              ON cl.target_table  = u.source_table
             AND cl.target_column = u.source_column
            WHERE u.depth < 5
        )
        SELECT DISTINCT
            u.*,
            CASE WHEN rc.rule_column IS NOT NULL THEN 'Y' ELSE 'N' END AS is_rule_col
        FROM upstream u
        LEFT JOIN rule_cols rc ON u.target_column = rc.rule_column
    """)

print("✅  Data functions ready")


In [ ]:
# ── Cell 4 : Graph-builder (verbatim copy from insight_lineage_tab.py) ──

_NODE_W, _NODE_H = 230, 54
_COL_GAP, _ROW_GAP, _LEFT_PAD = 340, 130, 80

def _safe_str(v) -> str:
    return str(v) if v is not None and pd.notna(v) else ""

def _node_ntype_insight(table: str, schema: str, rule_cols_in_table: set) -> str:
    t = (table  or "").lower()
    s = (schema or "").lower()
    if rule_cols_in_table:                            return "rule"
    if t.endswith("_ikg"):                            return "ikg"
    if "ikg" in s or "model" in s or "nlg" in s:     return "ikg"
    if t.startswith("temp_") or t.endswith("_tmp"):  return "src"
    return "ext"

def build_insight_lineage_payload(insight_type: str):
    insight_type = _safe_str(insight_type).strip()

    df      = get_insight_column_lineage(insight_type)
    df_meta = get_insight_rule_meta(insight_type)

    rule_cols = []
    if not df_meta.empty and "rule_column" in df_meta.columns:
        rule_cols = (df_meta["rule_column"].dropna().astype(str)
                     .str.strip().unique().tolist())
    rule_col_set = {c.lower() for c in rule_cols if c}

    if df.empty:
        return [], [], rule_cols, {"rule":len(rule_cols),"ikg":0,"src":0,"relations":0}

    edge_tuples = []
    for r in df.itertuples(index=False):
        st  = _safe_str(getattr(r,"source_table",None))
        sc2 = _safe_str(getattr(r,"source_column",None))
        tt  = _safe_str(getattr(r,"target_table",None))
        tc  = _safe_str(getattr(r,"target_column",None))
        lg  = _safe_str(getattr(r,"logic",None))
        pr  = _safe_str(getattr(r,"process",None))
        if st and tt: edge_tuples.append((st,sc2,tt,tc,lg,pr))

    if not edge_tuples:
        return [], [], rule_cols, {"rule":len(rule_cols),"ikg":0,"src":0,"relations":0}

    schema_map = {}
    for cp in [("source_table","source_schema"),("target_table","target_schema")]:
        tc2,sc3 = cp
        if tc2 in df.columns and sc3 in df.columns:
            for r in df[[tc2,sc3]].drop_duplicates().itertuples(index=False):
                tb,sh = _safe_str(r[0]),_safe_str(r[1])
                if tb and tb not in schema_map: schema_map[tb]=sh

    table_rule_cols = {}
    for st,sc2,tt,tc,*_ in edge_tuples:
        if tc.lower()  in rule_col_set: table_rule_cols.setdefault(tt,set()).add(tc)
        if sc2.lower() in rule_col_set: table_rule_cols.setdefault(st,set()).add(sc2)

    all_tables = {st for st,*_ in edge_tuples} | {et[2] for et in edge_tuples}
    seed_tables = {t for t in all_tables if table_rule_cols.get(t)}
    if not seed_tables:
        seed_tables = {et[2] for et in edge_tuples}

    adj_up = {}
    for st,sc2,tt,tc,*_ in edge_tuples:
        adj_up.setdefault(tt,[]).append(st)

    depths = {t:0 for t in seed_tables}
    queue  = list(seed_tables)
    visited= set(seed_tables)
    while queue:
        tbl = queue.pop(0)
        for src in adj_up.get(tbl,[]):
            if src not in visited:
                visited.add(src)
                depths[src] = depths[tbl] - 1
                queue.append(src)
    for t in all_tables:
        if t not in depths: depths[t] = 0

    depth_buckets = {}
    for tbl,d in depths.items():
        depth_buckets.setdefault(d,[]).append(tbl)
    min_depth = min(depth_buckets)

    nodes = []
    for depth in sorted(depth_buckets):
        ci = depth - min_depth
        for ri,tbl in enumerate(sorted(depth_buckets[depth])):
            x,y      = _LEFT_PAD+ci*_COL_GAP, 60+ri*_ROW_GAP
            rc_in    = sorted(table_rule_cols.get(tbl,set()))
            nt       = _node_ntype_insight(tbl, schema_map.get(tbl,""), set(rc_in))
            nodes.append({"id":tbl,"x":x,"y":y,"w":_NODE_W,"h":_NODE_H,
                           "ntype":nt,"rule_cols":rc_in,"level":ci})

    all_ids = {n["id"] for n in nodes}
    seen,edges = set(),[]
    for st,sc2,tt,tc,logic,proc in edge_tuples:
        if st not in all_ids or tt not in all_ids: continue
        k=(st,sc2,tt,tc)
        if k in seen: continue
        seen.add(k)
        edges.append({"from":st,"to":tt,"src_col":sc2,"tgt_col":tc,
                      "logic":logic if logic.lower() not in ("","none","nan") else "","process":proc})

    counts = {
        "rule":      sum(1 for n in nodes if n["ntype"]=="rule"),
        "ikg":       sum(1 for n in nodes if n["ntype"]=="ikg"),
        "src":       sum(1 for n in nodes if n["ntype"] in ("src","ext")),
        "relations": len(edges),
    }
    return nodes, edges, rule_cols, counts

print("✅  Graph builder ready")


In [ ]:
# ── Cell 5 : Template renderer ──────────────────────────────────────────

_SEARCH_PATHS = [
    Path("utils/insight_lineage_template.html"),
    Path("../utils/insight_lineage_template.html"),
    Path("insights_metadata_dashboard/utils/insight_lineage_template.html"),
]

def _find_template(paths):
    for p in paths:
        if p.exists(): return p
    raise FileNotFoundError(
        "insight_lineage_template.html not found. "
        f"Searched: {[str(p) for p in paths]}."
    )

TEMPLATE_PATH = _find_template(_SEARCH_PATHS)
print(f"✅  Template found: {TEMPLATE_PATH.resolve()}")

def render_insight_lineage_html(insight_type: str) -> str:
    template = TEMPLATE_PATH.read_text(encoding="utf-8")
    nodes, edges, rule_cols, counts = build_insight_lineage_payload(insight_type)
    payload = (
        f"const NODES      = {json.dumps(nodes)};\n"
        f"const EDGES      = {json.dumps(edges)};\n"
        f"const INSIGHT    = {json.dumps(insight_type)};\n"
        f"const RULE_COLS  = {json.dumps(rule_cols)};\n"
        f"const IL_COUNTS  = {json.dumps(counts)};"
    )
    html_out = re.sub(
        r"const NODES\s*=\s*\[\s*\];\s*"
        r"const EDGES\s*=\s*\[\s*\];\s*"
        r"const INSIGHT\s*=\s*\".*?\";\s*"
        r"const RULE_COLS\s*=\s*\[\s*\];\s*"
        r"const IL_COUNTS\s*=\s*\{.*?\};",
        payload, template, flags=re.DOTALL,
    )
    html_out = re.sub(r"<title>.*?</title>",
                      f"<title>{insight_type} — Insight Lineage</title>",
                      html_out, count=1, flags=re.DOTALL)
    return html_out

print("✅  Renderer ready")


In [ ]:
# ── Cell 6 : Load insight types for dropdown ────────────────────────────
print("Loading all insight types from DB …")
_df_it = get_all_insight_types()
_insight_list = _df_it["insight_type"].dropna().astype(str).tolist()
print(f"✅  {len(_insight_list)} insight types loaded")


In [ ]:
# ── Cell 7 : Interactive widgets ────────────────────────────────────────

_style  = {"description_width": "120px"}
_layout = widgets.Layout(width="520px")

w_insight = widgets.Dropdown(
    options     = _insight_list,
    description = "Insight Type:",
    layout      = _layout,
    style       = _style,
)
w_rule_cols_out = widgets.Output()   # shows rule-column chips
w_btn  = widgets.Button(
    description  = "Show Lineage ▶",
    button_style = "primary",
    layout       = widgets.Layout(width="160px", margin="8px 0 0 0"),
)
w_out  = widgets.Output()

def _on_insight_change(change):
    val = change["new"]
    w_rule_cols_out.clear_output()
    if not val: return
    try:
        df_m = get_insight_rule_meta(val)
    except Exception as e:
        with w_rule_cols_out: print(f"⚠  {e}")
        return
    with w_rule_cols_out:
        if not df_m.empty and "rule_column" in df_m.columns:
            rcs = df_m["rule_column"].dropna().astype(str).str.strip().unique().tolist()
            chips = "  ".join(f"[◈ {rc}]" for rc in sorted(rcs))
            print(f"ODM rule columns ({len(rcs)}):  {chips}")
        else:
            print("(no rule_column data found)")

def _on_btn_click(b):
    w_out.clear_output()
    it = w_insight.value
    if not it:
        with w_out: print("⚠  Please select an insight type.")
        return
    with w_out:
        print(f"Building lineage for insight: {it} …")
    try:
        html_content = render_insight_lineage_html(it)
    except Exception as exc:
        with w_out:
            w_out.clear_output(wait=True)
            print(f"❌  Error: {exc}")
        return
    tmp = Path(tempfile.mktemp(suffix=".html", dir="."))
    tmp.write_text(html_content, encoding="utf-8")
    with w_out:
        w_out.clear_output(wait=True)
        display(IFrame(src=str(tmp), width="100%", height="720px"))

w_insight.observe(_on_insight_change, names="value")
w_btn.on_click(_on_btn_click)

# Trigger initial rule-col load
if _insight_list:
    _on_insight_change({"new": w_insight.value})

display(widgets.VBox([
    widgets.HTML("<b>Insight Lineage Explorer</b>"),
    w_insight,
    w_rule_cols_out,
    w_btn,
    w_out,
]))


In [ ]:
# ── Cell 8 : Quick non-widget test (optional) ───────────────────────────
# Uncomment and run to check a specific insight_type without the widgets.

# INSIGHT_TO_TEST = "SOME_INSIGHT_TYPE_HERE"
# nodes, edges, rule_cols, counts = build_insight_lineage_payload(INSIGHT_TO_TEST)
# print(f"Nodes: {len(nodes)}  Edges: {len(edges)}  Rule cols: {rule_cols}")
# print(f"Counts: {counts}")
#
# # Show node types
# import collections
# by_type = collections.Counter(n['ntype'] for n in nodes)
# print(f"Node types: {dict(by_type)}")
print("Cell 8 ready — uncomment the lines above to run a headless test")


In [ ]:
# ── Cell 9 : Print raw data preview (optional) ──────────────────────────
# Useful for debugging the DB query output before rendering.

# INSIGHT_TO_TEST = "SOME_INSIGHT_TYPE_HERE"
# df_raw = get_insight_column_lineage(INSIGHT_TO_TEST)
# print(f"Rows returned: {len(df_raw)}")
# display(df_raw.head(20))
print("Cell 9 ready — uncomment to preview raw lineage data")


---
### Notes
- **Purple nodes** (◈ Rule) are the IKG tables whose columns are directly
  referenced by the ODM rule.
- **Blue nodes** are intermediate IKG pipeline tables.
- **Green/amber nodes** are source/EDW tables with no IKG upstream.
- The side panel (click any node) shows upstream/downstream tables and
  which rule columns pass through each node.
- Temp HTML files are written to the notebook directory — safe to delete.
